# Step 08.4 — Small MLX RNN Cell：把 RNN 参数正式交给 `nn.Module` 管理

8.2–8.3 已经完成了两件事：

```text
单步公式：h_t = tanh(W_x x_t + W_h h_(t-1) + b)
时间循环：for t in range(T) ...
```

但目前 `W_x / W_h / b` 仍是手工传入的 arrays。现在把它们正式封装到 Apple MLX 的 `nn.Module` 中，让模型自己拥有参数。

本节五部分：① 为什么需要 RNNCell Module；② 用两个 `nn.Linear` 精确实现公式并追踪 shape；③ 检查 parameter tree 与参数量；④ 在时间循环中反复调用同一个 cell，证明参数共享；⑤ 明确 runtime hidden state、trainable parameters 与未来 BPTT 的关系。

这一节仍然不做 optimizer/training；目标是先得到一个结构正确、参数可被 MLX 发现的最小 RNN Cell。


## 1. 从函数到 Module：区别不是“写成 class 比较好看”

8.3 中的函数：

```python
def rnn_step(x_t, h_prev, W_x, W_h, b):
    ...
```

要求调用者每一次都把参数显式传进去。它能计算，但模型参数并没有被组织成一个可递归发现的 parameter tree。

而 `nn.Module` 的职责是：

- 持有 trainable parameters；
- 持有子层（submodules）；
- 通过 `parameters()` 暴露参数树；
- 让后续 autodiff / optimizer 能统一处理这些参数；
- 保证同一个 Module 对象在多个时间步重复使用同一套参数。

因此 RNN Cell 的自然接口应该变成：

```text
cell(x_t, h_prev) -> h_t
```

权重已经属于 `cell` 自己，不再每次从外面传入。

同时要区分：`h_prev` 是**某个具体 sequence 在运行时的 state**，不是模型固定的 trainable parameter。


## 2. 用两个 `nn.Linear` 精确实现 Vanilla RNN 公式

我们希望实现：

$$
h_t = \tanh(W_xx_t + W_hh_{t-1} + b).
$$

用 MLX 的 row-vector 约定：

```text
x_t      : (B,C)
h_prev   : (B,H)

x_proj   : Linear(C,H,bias=True)
h_proj   : Linear(H,H,bias=False)
```

`x_proj` 负责 `x_t @ W_x.T + b`；`h_proj` 只负责 `h_prev @ W_h.T`。

为什么第二条 Linear 设置 `bias=False`？因为公式中只有**一个** bias `b`。如果两个 Linear 都带 bias，就会实现成 `... + b_x + b_h`，已经不是我们当前定义的最小 Vanilla RNN 公式。

下面正式实现。


In [ ]:
import mlx.core as mx
import mlx.nn as nn

class RNNCell(nn.Module):
    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size

        # W_x and b
        self.x_proj = nn.Linear(input_size, hidden_size, bias=True)

        # W_h, deliberately without a second bias
        self.h_proj = nn.Linear(hidden_size, hidden_size, bias=False)

    def __call__(self, x_t, h_prev):
        preactivation = self.x_proj(x_t) + self.h_proj(h_prev)
        return mx.tanh(preactivation)

C, H, B = 3, 2, 4
cell = RNNCell(C, H)

x_t = mx.ones((B, C))
h_prev = mx.zeros((B, H))
h_t = cell(x_t, h_prev)
mx.eval(h_t)

print('x_t shape   :', x_t.shape)
print('h_prev shape:', h_prev.shape)
print('h_t shape   :', h_t.shape)


### 逐层 Shape

`x_proj`：

```text
(B,C) @ (C,H) -> (B,H)
```

MLX 实际存储 `weight.shape=(H,C)`，所以 API 内部等价于 `x_t @ weight.T`。

`h_proj`：

```text
(B,H) @ (H,H) -> (B,H)
```

两条路径 shape 相同才能相加，然后 `tanh` 不改变 shape，因此最终仍是 `(B,H)`。

`input_size`、`hidden_size` 是普通 Python integers，用来描述配置；真正可训练的是 Linear 里的 `mx.array` parameters。


## 3. Parameter Tree：MLX 到底发现了哪些参数？

我们预期这个 Cell 有：

```text
x_proj.weight : (H,C) -> H*C
x_proj.bias   : (H,)  -> H
h_proj.weight : (H,H) -> H*H
```

总参数量：

$$
P = HC + H + H^2.
$$

这正好与 8.2 手算得到的 Vanilla RNN 参数量一致。

现在不靠猜，直接让 MLX 把 parameter tree 展平。


In [ ]:
from mlx.utils import tree_flatten

flat_params = tree_flatten(cell.parameters())

print('parameter leaves:')
for name, value in flat_params:
    print(f'  {name:24s} shape={value.shape} size={value.size}')

actual_count = sum(value.size for _, value in flat_params)
expected_count = H*C + H + H*H

print('\nactual parameter count  :', actual_count)
print('expected parameter count:', expected_count)
print('match:', actual_count == expected_count)


### 再把 Module 黑盒拆掉一次

和 Step 4.1 验证 `nn.Linear` 一样，我们可以直接使用 Module 里的真实随机参数做手工公式：

```text
manual = tanh(
    x_t @ x_proj.weight.T
    + x_proj.bias
    + h_prev @ h_proj.weight.T
)
```

它应该与 `cell(x_t,h_prev)` 完全一致。


In [ ]:
via_cell = cell(x_t, h_prev)

manual = mx.tanh(
    x_t @ cell.x_proj.weight.T
    + cell.x_proj.bias
    + h_prev @ cell.h_proj.weight.T
)

mx.eval(via_cell, manual)

print('module output shape:', via_cell.shape)
print('manual output shape:', manual.shape)
print('module == manual   :', bool(mx.allclose(via_cell, manual)))


这个验证很重要：`nn.Module` 没有增加新的 RNN 数学，它只是把参数组织、复用和发现机制封装起来。

### 参数量中哪一部分增长最快？

固定 input width C 时，recurrent matrix `W_h` 有 `H^2` 个参数，所以 H 增大时这部分会二次增长。

下面用小图只观察参数组成，不比较模型性能。


In [ ]:
import matplotlib.pyplot as plt

C_plot, H_plot = 4, 8
parts = {
    'W_x: H*C': H_plot*C_plot,
    'W_h: H^2': H_plot*H_plot,
    'bias: H': H_plot,
}

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(list(parts.keys()), list(parts.values()))
ax.set_ylabel('parameter count')
ax.set_title(f'Vanilla RNN Cell parameter components (C={C_plot}, H={H_plot})')
plt.show()


`ax.bar()` 适合比较离散组成部分。这里可以直接看到，hidden width H 增大时 `W_h` 很快成为主要参数来源之一。

但参数量只是资源的一部分；真正沿 T 展开以后，还有 activation 与 BPTT 的内存/计算成本。


## 4. 时间循环里必须反复调用“同一个 cell”，而不是每个 t 新建一个

RNN 的核心假设之一是 **time-shared parameters**。

正确结构：

```python
cell = RNNCell(C,H)

for t in range(T):
    h = cell(X[:,t,:], h)
```

这里每个时间步调用的都是**同一个 Python Module 对象**，因此使用同一套 `x_proj.weight / x_proj.bias / h_proj.weight`。

错误思路是：

```python
for t in range(T):
    cell_t = RNNCell(C,H)   # 每一步重新随机创建参数
    h = cell_t(X[:,t,:], h)
```

那会变成每个位置拥有一套完全不同的参数，既破坏经典 RNN 的时间参数共享，也会让 parameter count 随 T 增长。


In [ ]:
def run_sequence(cell, X):
    B_local, T_local, _ = X.shape
    h = mx.zeros((B_local, cell.hidden_size))
    states = []

    for t in range(T_local):
        h = cell(X[:, t, :], h)
        states.append(h)

    return mx.stack(states, axis=1), h

B, T, C, H = 2, 4, 3, 2
X = mx.arange(B*T*C, dtype=mx.float32).reshape(B, T, C) / 10.0

cell = RNNCell(C, H)
H_all, h_final = run_sequence(cell, X)
mx.eval(H_all, h_final)

print('X shape      :', X.shape)
print('H_all shape  :', H_all.shape)
print('h_final shape:', h_final.shape)

count_before = sum(v.size for _, v in tree_flatten(cell.parameters()))
_ = run_sequence(cell, X[:, :2, :])  # same cell, shorter T
count_after = sum(v.size for _, v in tree_flatten(cell.parameters()))

print('parameter count before:', count_before)
print('parameter count after :', count_after)
print('independent of T      :', count_before == count_after)


注意：`parameter count independent of T` 只表示模型拥有的权重数不随序列长度增加。

它**不表示**运行更长序列没有额外代价。T 增大仍然会增加 forward 计算和训练时需要展开的计算图。

还有一个重要区别：不同 batch rows 共享这套参数，但 `h[b]` 是各自独立的 runtime state，不会因为参数共享就互相混在一起。


## 5. 为什么现在它才真正“可训练”？以及哪些东西仍然不是 Parameter

当前 `RNNCell` 已满足一个可训练 MLX Module 的关键条件：

```text
cell.parameters()
↓
能递归找到 W_x、W_h、b
```

后续定义 loss 后，MLX autodiff 可以沿时间展开的计算图，把来自多个时间步的梯度贡献最终汇总到**同一套共享 parameters**。这就是 Step 9.1 要讲的 Backpropagation Through Time（BPTT）的入口。

但必须严格区分三类东西：

```text
W_x / W_h / b       -> trainable parameters
x_t / X             -> input data / activations
h_prev / h_t        -> runtime hidden states / activations
```

初始零状态 `mx.zeros((B,H))` 不是 trainable parameter。我们也不把某个 batch 的 evolving `h_t` 永久挂成 Module parameter，因为它属于当前 sequence 的运行状态。

同样，当前这个 RNNCell 只完成 hidden-state transition；它没有语言模型输出层，也还没有 loss / optimizer。不要把“parameter tree 已准备好”误解成“模型已经训练”。

### 8.4 最终结论

- `RNNCell(nn.Module)` 的价值是拥有并组织参数，而不仅仅是换一种代码写法。
- 用 `Linear(C,H,bias=True)` + `Linear(H,H,bias=False)` 可以精确对应 `W_x x_t + W_h h_prev + b`，避免重复 bias。
- Cell 的输入输出契约是 `(B,C) + (B,H) -> (B,H)`。
- `parameters()` 能发现 `W_x / W_h / b`，参数量仍是 `HC + H^2 + H`。
- 用真实 Module 参数手算得到的结果应与 `cell(x_t,h_prev)` 一致，说明封装没有改变数学。
- 时间循环必须复用同一个 `cell` 对象，经典 RNN 才实现跨时间 parameter sharing。
- 参数共享不等于 hidden-state sharing；不同 batch rows 有各自的 state。
- `h_t` 是 runtime activation，不是 parameter；未来 BPTT 会让多个时间步对同一参数产生梯度贡献。
- 下一节 8.5：总结 RNN 到底解决了哪些 sequence 问题、代价在哪里，并为 Step 9 的 BPTT / vanishing-gradient / LSTM-GRU 建立问题清单。
